librerias necesarias

In [1]:
!pip install spacy trafilatura pandas matplotlib seaborn plotly wordcloud gradio -q
!python -m spacy download es_core_news_sm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 60.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import trafilatura
import pandas as pd
import re
import os
import requests
from bs4 import BeautifulSoup

In [3]:
# ── Función de scraping con headers de navegador ─────────────────────────────
import json
from datetime import datetime

def extraer_noticias_web(urls_cenital, urls_infobae):
    """
    Extrae artículos de Cenital e Infobae usando requests con headers
    para evitar bloqueos, y trafilatura para limpiar el texto.
    Devuelve lista con todas las columnas requeridas por el TPI 2.
    """

    HEADERS = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "es-AR,es;q=0.9,en;q=0.8",
        "Accept-Encoding": "gzip, deflate",   # sin zstd
        "Connection": "keep-alive",
    }

    articulos = []
    todas = [("cenital", u) for u in urls_cenital] + [("infobae", u) for u in urls_infobae]

    for i, (grupo, url) in enumerate(todas):
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            response.encoding = "utf-8"
            html = response.text

            resultado_json = trafilatura.extract(
                html,
                output_format="json",
                include_comments=False,
                include_tables=False,
                favor_precision=True,
            )

            if resultado_json:
                datos = json.loads(resultado_json)
                texto = datos.get("text", "").strip()
                titulo = datos.get("title", "").strip()
                fecha  = datos.get("date", "")
                autor  = datos.get("author", "")
            else:
                texto = ""

            if not texto or len(texto) < 200:
                print(f"✗ Sin texto útil: {url}")
                continue

            # id único por grupo
            idx = sum(1 for a in articulos if a["grupo_comparacion"] == grupo) + 1

            articulos.append({
                "id":                 f"{grupo}_{idx:02d}",
                "fecha":              fecha,
                "medio":              grupo.capitalize(),
                "autor":              autor,
                "titulo":             titulo,
                "texto":              texto,
                "grupo_comparacion":  grupo,
                "url":                url,
            })
            print(f"✓ [{grupo}] {titulo[:65]}...")

        except Exception as e:
            print(f"✗ Error con {url}: {e}")

    return articulos

In [4]:
# ── URLs por grupo ────────────────────────────────────────────────────────────
urls_cenital = [
    "https://cenital.com/la-burbuja-de-la-inteligencia-artificial/",
    "https://cenital.com/la-inteligencia-artificial-no-muestra-el-pasado-lo-reescribe/",
    "https://cenital.com/elon-musk-la-quiere-detener-y-a-la-vez-desarrollar-los-peligros-de-liberar-o-dominar-la-inteligencia-artificial/",
    "https://cenital.com/changas-digitales-que-hay-tras-la-cortina-de-las-nuevas-maneras-de-ganar-dinero-en-internet/",
    "https://cenital.com/debi-preguntarle-a-la-inteligencia-artificial/",
]

urls_infobae = [
    "https://www.infobae.com/tecno/2026/03/30/la-ia-cambiara-el-empleo-estos-son-los-dos-perfiles-con-mas-futuro/",
    "https://www.infobae.com/tecno/2026/02/23/inteligencia-artificial-y-trabajo-la-ventaja-que-mas-vale-en-2026-no-la-da-ninguna-empresa-ni-ningun-curso/",
    "https://www.infobae.com/tecno/2025/07/17/la-inteligencia-artificial-acelera-despidos-y-redefine-el-empleo-en-el-sector-tecnologico/",
    "https://www.infobae.com/tecno/2025/11/04/las-10-profesiones-que-la-inteligencia-artificial-podria-reemplazar-en-la-proxima-decada-segun-microsoft/",
    "https://www.infobae.com/tecno/2025/07/01/la-inteligencia-artificial-esta-eliminando-empleos-para-quienes-buscan-su-primer-empleo/",
]

In [5]:
# ── Ejecutar scraping ─────────────────────────────────────────────────────────
datos = extraer_noticias_web(urls_cenital, urls_infobae)
df = pd.DataFrame(datos)
print(f"\nTotal extraídos: {len(df)} artículos")
print(df.groupby("grupo_comparacion")["id"].count())
df.head()

✓ [cenital] ...
✓ [cenital] ...
✓ [cenital] ...
✓ [cenital] ...
✓ [cenital] ...
✓ [infobae] ...
✓ [infobae] ...
✓ [infobae] ...
✓ [infobae] ...
✓ [infobae] ...

Total extraídos: 10 artículos
grupo_comparacion
cenital    5
infobae    5
Name: id, dtype: int64


,id,fecha,medio,autor,titulo,texto,grupo_comparacion,url
0,cenital_01,,Cenital,,,Se proyecta que las empresas de tecnología est...,cenital,https://cenital.com/la-burbuja-de-la-inteligen...
1,cenital_02,,Cenital,,,En febrero de 2024 OpenAI presentó en sociedad...,cenital,https://cenital.com/la-inteligencia-artificial...
2,cenital_03,,Cenital,,,"En marzo de 2022, Elon Musk pidió detener la i...",cenital,https://cenital.com/elon-musk-la-quiere-detene...
3,cenital_04,,Cenital,,,La transición de las páginas web a la economía...,cenital,https://cenital.com/changas-digitales-que-hay-...
4,cenital_05,,Cenital,,,"El autor de esta nota es Sebastián Ceria, miem...",cenital,https://cenital.com/debi-preguntarle-a-la-inte...


In [6]:
df.iloc[0]["texto"][:500]

'Se proyecta que las empresas de tecnología estadounidenses gastarán sólo este año más de 500 mil millones de dólares en inteligencia artificial, una cifra que no guarda demasiada relación con las ganancias que produce. Por eso, y desde hace largos meses, suena recurrentemente la misma alarma: la posibilidad de que estemos ante otra burbuja económica y, peor, la de qué sucederá cuando estalle.\nEn apenas unos pocos años, el asunto de la IA pasó de ser un tema de nicho con el que se coqueteaba desd'

In [7]:
# ── Auditoría del corpus extraído ─────────────────────────────────────────────
print(f"Total artículos: {len(df)}")
print()
print(df.groupby("grupo_comparacion")["id"].count())
print()
print(df[["id", "grupo_comparacion", "titulo", "fecha"]].to_string(index=False))
print()
print("Largo de textos (palabras aproximadas):")
df["n_palabras"] = df["texto"].apply(lambda x: len(x.split()))
print(df[["id", "n_palabras"]].to_string(index=False))

Total artículos: 10

grupo_comparacion
cenital    5
infobae    5
Name: id, dtype: int64

        id grupo_comparacion titulo fecha
cenital_01           cenital             
cenital_02           cenital             
cenital_03           cenital             
cenital_04           cenital             
cenital_05           cenital             
infobae_01           infobae             
infobae_02           infobae             
infobae_03           infobae             
infobae_04           infobae             
infobae_05           infobae             

Largo de textos (palabras aproximadas):
        id  n_palabras
cenital_01        1696
cenital_02        1509
cenital_03        2460
cenital_04        2001
cenital_05        1854
infobae_01         662
infobae_02        1254
infobae_03         946
infobae_04         880
infobae_05         653


In [8]:
# ── Verificar campos críticos ──────────────────────────────────────────────────
print(df[["id", "titulo", "fecha", "autor"]].to_string(index=False))

        id titulo fecha autor
cenital_01                   
cenital_02                   
cenital_03                   
cenital_04                   
cenital_05                   
infobae_01                   
infobae_02                   
infobae_03                   
infobae_04                   
infobae_05                   


In [9]:
# ── Completar campos manualmente ──────────────────────────────────────────────

metadata = {
    "cenital_01": {
        "titulo": "La burbuja de la inteligencia artificial",
        "fecha":  "2025-10-16",
        "autor":  "Valentin Muro",
    },
    "cenital_02": {
        "titulo": "La inteligencia artificial no muestra el pasado, lo reescribe",
        "fecha":  "2026-04-23",
        "autor":  "Valentin Muro",
    },
    "cenital_03": {
        "titulo": "Elon Musk la quiere detener y a la vez desarrollar",
        "fecha":  "2023-04-01",
        "autor":  "Juan Zaragoza",
    },
    "cenital_04": {
        "titulo": "Changas digitales: qué hay detras de la cortina de las nuevas maneras de ganar dinero en internet",
        "fecha":  "2024-06-18",
        "autor":  "Irina Sternik",
    },
    "cenital_05": {
        "titulo": "¿Debí preguntarle a la inteligencia artificial?",
        "fecha":  "2025-02-02",
        "autor":  "Sebastián Ceria",
    },
    "infobae_01": {
        "titulo": "La IA cambiará el empleo: estos son los dos perfiles con más futuro",
        "fecha":  "2026-03-30",
        "autor":  "Santiago Neira",
    },
    "infobae_02": {
        "titulo": "Inteligencia artificial y trabajo: la ventaja que más vale en 2026",
        "fecha":  "2026-02-23",
        "autor":  "Opy Morales",
    },
    "infobae_03": {
        "titulo": "La inteligencia artificial acelera despidos y redefine el empleo en el sector tecnológico",
        "fecha":  "2025-06-15",
        "autor":  "Constanza Almirón",
    },
    "infobae_04": {
        "titulo": "Las 10 profesiones que la inteligencia artificial podría reemplazar en la próxima década, según Microsoft",
        "fecha":  "2025-11-04",
        "autor":  "Martina Cortés Moschetti",
    },
    "infobae_05": {
        "titulo": "La inteligencia artificial está eliminando empleos para quienes buscan su primer trabajo",
        "fecha":  "2025-06-01",
        "autor":  "Santiago Neira",
    },
}

for idx, row in df.iterrows():
    art_id = row["id"]
    if art_id in metadata:
        df.at[idx, "titulo"] = metadata[art_id]["titulo"]
        df.at[idx, "fecha"]  = metadata[art_id]["fecha"]
        df.at[idx, "autor"]  = metadata[art_id]["autor"]

print("Campos completados:")
print(df[["id", "titulo", "fecha", "autor"]].to_string(index=False))

Campos completados:
        id                                                                                                    titulo      fecha                    autor
cenital_01                                                                  La burbuja de la inteligencia artificial 2025-10-16            Valentin Muro
cenital_02                                             La inteligencia artificial no muestra el pasado, lo reescribe 2026-04-23            Valentin Muro
cenital_03                                                        Elon Musk la quiere detener y a la vez desarrollar 2023-04-01            Juan Zaragoza
cenital_04         Changas digitales: qué hay detras de la cortina de las nuevas maneras de ganar dinero en internet 2024-06-18            Irina Sternik
cenital_05                                                           ¿Debí preguntarle a la inteligencia artificial? 2025-02-02          Sebastián Ceria
infobae_01                                       La IA cambiar

In [10]:
# ── Exportar corpus como CSV ───────────────────────────────────────────────────

# Columnas en el orden que espera el TPI 2
columnas_finales = ["id", "fecha", "medio", "autor", "titulo", "texto", "grupo_comparacion", "url"]

df_final = df[columnas_finales].copy()

df_final.to_csv("corpus_tpi2.csv", index=False, encoding="utf-8-sig")

print(f"✓ corpus_tpi2.csv exportado con {len(df_final)} artículos")
print(f"  Columnas: {list(df_final.columns)}")
print(f"  Grupos: {df_final['grupo_comparacion'].value_counts().to_dict()}")

✓ corpus_tpi2.csv exportado con 10 artículos
  Columnas: ['id', 'fecha', 'medio', 'autor', 'titulo', 'texto', 'grupo_comparacion', 'url']
  Grupos: {'cenital': 5, 'infobae': 5}
